# HIV Bioactivity Paper – Statistical Comparison for 12 Models

This notebook produces the statistical comparison for the **12 models** in the manuscript.

Recommended statistical workflow:
1. ROC-AUC is used as the **primary metric**.
2. A **Friedman test** is first applied to check whether there is an overall difference among all models.
3. If the Friedman test is significant, post-hoc comparisons are performed between the best model (**GDL**) and the other 11 models using a **paired t-test**.
4. A **Holm correction** is applied to control multiple-comparison bias.
5. Effect size is reported using **paired Cohen's d**.

Note: This setup is the easiest to defend in the manuscript and yields the most readable results. Wilcoxon results can also be produced as supportive analyses if desired.


In [1]:
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, ttest_rel, wilcoxon, shapiro
from statsmodels.stats.multitest import multipletests
from IPython.display import display, Markdown

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.6f}')


In [2]:
models = {'GCN': {'fold': [1, 2, 3, 4, 5], 'accuracy': [0.698962, 0.730104, 0.730104, 0.726644, 0.709343], 'precision': [0.668539, 0.763359, 0.777778, 0.68254, 0.656863], 'recall': [0.809524, 0.680272, 0.662162, 0.871622, 0.905405], 'f1': [0.732308, 0.719424, 0.715328, 0.765579, 0.761364], 'roc_auc': [0.797595, 0.78581, 0.81129, 0.837694, 0.830698]}, 'GAT': {'fold': [1, 2, 3, 4, 5], 'accuracy': [0.653979, 0.698962, 0.66436, 0.685121, 0.692042], 'precision': [0.662069, 0.720588, 0.650888, 0.724409, 0.685535], 'recall': [0.653061, 0.666667, 0.743243, 0.621622, 0.736486], 'f1': [0.657534, 0.69258, 0.694006, 0.669091, 0.710098], 'roc_auc': [0.730622, 0.759749, 0.781052, 0.782682, 0.789774]}, 'MPNN': {'fold': [1, 2, 3, 4, 5], 'accuracy': [0.67128, 0.622837, 0.705882, 0.67474, 0.640138], 'precision': [0.680556, 0.613095, 0.700637, 0.655172, 0.648649], 'recall': [0.666667, 0.70068, 0.743243, 0.77027, 0.648649], 'f1': [0.67354, 0.653968, 0.721311, 0.708075, 0.648649], 'roc_auc': [0.725065, 0.676008, 0.756134, 0.783017, 0.705147]}, 'RNN': {'fold': [1, 2, 3, 4, 5], 'accuracy': [0.832335, 0.862275, 0.850299, 0.796407, 0.347305], 'precision': [0.451613, 0.571429, 0.526316, 0.366667, 0.162602], 'recall': [0.56, 0.32, 0.384615, 0.423077, 0.769231], 'f1': [0.5, 0.410256, 0.444444, 0.392857, 0.268456], 'roc_auc': [0.746197, 0.776338, 0.745499, 0.738953, 0.582651]}, 'GRU': {'fold': [1, 2, 3, 4, 5], 'accuracy': [0.9, 0.866667, 0.916667, 0.883333, 0.891213], 'precision': [0.9625, 0.858696, 0.975904, 0.858586, 0.891304], 'recall': [0.785714, 0.806122, 0.818182, 0.858586, 0.836735], 'f1': [0.865169, 0.831579, 0.89011, 0.858586, 0.863158], 'roc_auc': [0.923326, 0.916786, 0.951071, 0.943764, 0.91294]}, 'RF': {'fold': [1, 2, 3, 4, 5], 'accuracy': [0.851211, 0.882353, 0.858131, 0.913495, 0.820069], 'precision': [0.926829, 0.967213, 0.934959, 0.969231, 0.920354], 'recall': [0.77027, 0.797297, 0.777027, 0.857143, 0.707483], 'f1': [0.841328, 0.874074, 0.848708, 0.909747, 0.8], 'roc_auc': [0.9182, 0.948989, 0.925029, 0.949267, 0.895348]}, 'SVM': {'fold': [1, 2, 3, 4, 5], 'accuracy': [0.875433, 0.858131, 0.844291, 0.885813, 0.83737], 'precision': [0.905109, 0.878571, 0.875912, 0.907801, 0.863309], 'recall': [0.843537, 0.836735, 0.810811, 0.864865, 0.810811], 'f1': [0.873239, 0.857143, 0.842105, 0.885813, 0.836237], 'roc_auc': [0.936117, 0.909121, 0.890502, 0.928215, 0.912306]}, 'MLP': {'fold': [1, 2, 3, 4, 5], 'accuracy': [0.844291, 0.83391, 0.83737, 0.865052, 0.820069], 'precision': [0.892308, 0.902439, 0.863309, 0.875862, 0.863636], 'recall': [0.789116, 0.755102, 0.810811, 0.858108, 0.77027], 'f1': [0.837545, 0.822222, 0.836237, 0.866894, 0.814286], 'roc_auc': [0.913146, 0.864664, 0.902291, 0.921794, 0.896109]}, 'Mol2Vec+SVM': {'fold': [1, 2, 3, 4, 5], 'accuracy': [0.82669, 0.856153, 0.82149, 0.859619, 0.838821], 'precision': [0.904167, 0.927126, 0.887097, 0.911538, 0.876866], 'recall': [0.738095, 0.778912, 0.745763, 0.80339, 0.79661], 'f1': [0.812734, 0.84658, 0.810313, 0.854054, 0.834813], 'roc_auc': [0.893777, 0.914173, 0.892595, 0.923416, 0.912405]}, 'CNN': {'fold': [1, 2, 3, 4, 5], 'accuracy': [0.499133, 0.509532, 0.528596, 0.499133, 0.495667], 'precision': [0.506775, 0.516923, 0.536977, 0.508621, 0.506803], 'recall': [0.636054, 0.571429, 0.566102, 0.6, 0.505085], 'f1': [0.564103, 0.542811, 0.551155, 0.550544, 0.505942], 'roc_auc': [0.534026, 0.509964, 0.521842, 0.490251, 0.503835]}, 'BRNN': {'fold': [1, 2, 3, 4, 5], 'accuracy': [0.462738, 0.5026, 0.533795, 0.511265, 0.514731], 'precision': [0.475758, 0.509749, 0.540625, 0.511265, 0.517241], 'recall': [0.534014, 0.622449, 0.586441, 1.0, 0.762712], 'f1': [0.503205, 0.56049, 0.562602, 0.676606, 0.616438], 'roc_auc': [0.45303, 0.490493, 0.519762, 0.483171, 0.507898]}, 'GDL': {'fold': [1, 2, 3, 4, 5], 'accuracy': [0.858131, 0.83391, 0.878893, 0.885813, 0.899654], 'precision': [1.0, 1.0, 1.0, 0.982759, 1.0], 'recall': [0.707143, 0.690323, 0.744526, 0.786207, 0.811688], 'f1': [0.828452, 0.816794, 0.853556, 0.873563, 0.896057], 'roc_auc': [0.938351, 0.944872, 0.972964, 0.953113, 0.970899]}}


In [3]:

def summarize_models(models_dict, metric_cols=('accuracy', 'precision', 'recall', 'f1', 'roc_auc')):
    rows = []
    for model_name, data in models_dict.items():
        df = pd.DataFrame(data)
        row = {'Model': model_name}
        for m in metric_cols:
            vals = df[m].astype(float).values
            row[m] = f"{np.mean(vals):.4f} ± {np.std(vals, ddof=1):.4f}"
        rows.append(row)
    return pd.DataFrame(rows)


def cohens_d_paired(x, y):
    diff = np.array(x, dtype=float) - np.array(y, dtype=float)
    sd = np.std(diff, ddof=1)
    if np.isclose(sd, 0.0):
        return np.nan
    return np.mean(diff) / sd


def interpret_effect(d):
    ad = abs(d)
    if ad >= 2.0:
        return 'very strong'
    if ad >= 0.8:
        return 'strong'
    if ad >= 0.5:
        return 'orta'
    if ad >= 0.2:
        return 'small'
    return 'ihmal edilebilir'


def compare_against_base(models_dict, base_name='GDL', metric='roc_auc', alternative='greater'):
    base = np.array(models_dict[base_name][metric], dtype=float)
    rows = []
    for model_name, data in models_dict.items():
        if model_name == base_name:
            continue
        other = np.array(data[metric], dtype=float)
        diff = base - other

        t_res = ttest_rel(base, other, alternative=alternative)
        shapiro_p = shapiro(diff).pvalue
        try:
            wilcoxon_p = wilcoxon(base, other, alternative=alternative).pvalue
        except Exception:
            wilcoxon_p = np.nan

        rows.append({
            'Comparison': f'{base_name} vs {model_name}',
            'Metric': metric.upper(),
            'Base Mean': float(np.mean(base)),
            'Other Mean': float(np.mean(other)),
            'Mean Diff': float(np.mean(diff)),
            't-test p': float(t_res.pvalue),
            'Shapiro p': float(shapiro_p),
            'Normality OK': 'Yes' if shapiro_p > 0.05 else 'No',
            'Wilcoxon p': float(wilcoxon_p) if not np.isnan(wilcoxon_p) else np.nan,
            "Cohen's d": float(cohens_d_paired(base, other)),
        })

    df = pd.DataFrame(rows).sort_values('Mean Diff', ascending=False).reset_index(drop=True)
    reject, p_holm, _, _ = multipletests(df['t-test p'].values, alpha=0.05, method='holm')
    df['Holm-corrected p'] = p_holm
    df['Significant after Holm'] = np.where(reject, 'Yes', 'No')
    df['Effect'] = df["Cohen's d"].map(interpret_effect)
    df['Result'] = np.where(
        df['Significant after Holm'].eq('Yes'),
        'Significant',
        np.where(df["Cohen's d"].abs() >= 0.8, 'Statistically borderline / practically strong', 'Not significant')
    )
    return df


def format_for_paper(df):
    out = df.copy()
    out['Base Mean'] = out['Base Mean'].map(lambda x: f'{x:.4f}')
    out['Other Mean'] = out['Other Mean'].map(lambda x: f'{x:.4f}')
    out['Mean Diff'] = out['Mean Diff'].map(lambda x: f'{x:+.4f}')
    out['t-test p'] = out['t-test p'].map(lambda x: f'{x:.6f}')
    out['Holm-corrected p'] = out['Holm-corrected p'].map(lambda x: f'{x:.6f}')
    out['Shapiro p'] = out['Shapiro p'].map(lambda x: f'{x:.6f}')
    out['Wilcoxon p'] = out['Wilcoxon p'].map(lambda x: 'NA' if pd.isna(x) else f'{x:.6f}')
    out["Cohen's d"] = out["Cohen's d"].map(lambda x: f'{x:.2f}')
    return out


## 1. Summary performance table for 12 models


In [ ]:
summary_df = summarize_models(models)
summary_df


## 2. Omnibus test: is there an overall difference among all models?

This cell runs the **Friedman test** on ROC-AUC. If p < 0.05, there is an overall significant difference among models and post-hoc comparison follows.


In [ ]:
roc_lists = [np.array(v['roc_auc'], dtype=float) for v in models.values()]
model_names = list(models.keys())
friedman_stat, friedman_p = friedmanchisquare(*roc_lists)
print(f'Friedman statistic = {friedman_stat:.6f}')
print(f'Friedman p-value   = {friedman_p:.8f}')
if friedman_p < 0.05:
    print('Result: There is an overall significant difference among models in terms of ROC-AUC.')
else:
    print('Result: There is no overall significant difference among models in terms of ROC-AUC.')


## 3. Post-hoc test: comparison of GDL with the other 11 models

Here the main hypothesis is directional: **GDL yields higher ROC-AUC than the other models.**

Therefore a paired **one-sided paired t-test** was used. A **Holm correction** was applied for multiple comparisons.

Wilcoxon and Shapiro results are reported as additional checks; reporting t-test + Holm + Cohen's d in the main manuscript table is sufficient.


In [ ]:
raw_stats = compare_against_base(models, base_name='GDL', metric='roc_auc', alternative='greater')
raw_stats


In [ ]:
paper_table = format_for_paper(raw_stats)[['Comparison', 'Metric', 'Base Mean', 'Other Mean', 'Mean Diff', 't-test p', 'Holm-corrected p', "Cohen's d", 'Effect', 'Significant after Holm', 'Result']]
paper_table


## 4. Short commentary draft for the manuscript

This cell produces a short automatic summary that can be turned directly into manuscript text.


In [ ]:

sig = raw_stats[raw_stats['Significant after Holm'] == 'Yes']['Comparison'].tolist()
not_sig = raw_stats[raw_stats['Significant after Holm'] == 'No']['Comparison'].tolist()

print('Friedman p-value:', f'{friedman_p:.8f}')
print()
print('Comparisons significant after Holm correction:')
for s in sig:
    print('-', s)
print()
print('Comparisons not significant after Holm correction:')
for s in not_sig:
    print('-', s)


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Build a long-format dataframe from the models dictionary
boxplot_rows = []
for model_name, model_data in models.items():
    for fold_id, auc_val in zip(model_data['fold'], model_data['roc_auc']):
        boxplot_rows.append({
            'Model': model_name,
            'Fold': fold_id,
            'ROC_AUC': float(auc_val)
        })

boxplot_df = pd.DataFrame(boxplot_rows)

plt.figure(figsize=(12, 6))
sns.boxplot(data=boxplot_df, x='Model', y='ROC_AUC')
sns.stripplot(data=boxplot_df, x='Model', y='ROC_AUC', dodge=False, alpha=0.7)
plt.title('ROC-AUC Distribution Across Models (5-Fold)')
plt.xlabel('Model')
plt.ylabel('ROC-AUC')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# paper_table or raw_stats can be used; raw_stats is safer here
heatmap_df = raw_stats[['Comparison', 'Holm-corrected p']].copy()
heatmap_df['Base'] = heatmap_df['Comparison'].apply(lambda x: x.split(' vs ')[0])
heatmap_df['Other'] = heatmap_df['Comparison'].apply(lambda x: x.split(' vs ')[1])

heatmap_matrix = heatmap_df.pivot(index='Base', columns='Other', values='Holm-corrected p')

plt.figure(figsize=(12, 3))
sns.heatmap(heatmap_matrix, annot=True, cmap='coolwarm_r', fmt='.4f', cbar_kws={'label': 'Holm-corrected p'})
plt.title("Holm-Corrected p-values for GDL vs Other Models")
plt.xlabel("Compared Model")
plt.ylabel("Base Model")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

effect_df = raw_stats[['Comparison', "Cohen's d"]].copy()

plt.figure(figsize=(12, 5))
sns.barplot(data=effect_df, x='Comparison', y="Cohen's d")
plt.title("Effect Size for GDL vs Other Models")
plt.xlabel("Comparison")
plt.ylabel("Cohen's d")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata

# Compute ranks so that larger ROC-AUC = better for each fold
rank_rows = []
model_names = list(models.keys())

fold_count = len(next(iter(models.values()))['roc_auc'])

for i in range(fold_count):
    fold_scores = [models[m]['roc_auc'][i] for m in model_names]
    fold_ranks = rankdata([-x for x in fold_scores], method='average')  # larger value is better
    for m, r in zip(model_names, fold_ranks):
        rank_rows.append({'Model': m, 'Fold': i + 1, 'Rank': r})

rank_df = pd.DataFrame(rank_rows)
mean_rank_df = rank_df.groupby('Model', as_index=False)['Rank'].mean().sort_values('Rank')

plt.figure(figsize=(10, 5))
sns.barplot(data=mean_rank_df, x='Model', y='Rank')
plt.title('Mean Model Ranks Based on ROC-AUC Across 5 Folds')
plt.xlabel('Model')
plt.ylabel('Mean Rank (Lower is Better)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# assign colors based on Cohen's d
colors = []
for val in raw_stats["Cohen's d"]:
    if val > 2:
        colors.append("red")        # very strong
    elif val > 1:
        colors.append("orange")     # strong
    else:
        colors.append("yellow")  # weak

plt.figure(figsize=(12, 5))

ax = sns.barplot(
    data=raw_stats,
    x='Comparison',
    y="Cohen's d",
    palette=colors
)

# write values
for i, v in enumerate(raw_stats["Cohen's d"]):
    ax.text(i, v + 0.5, f"{v:.2f}", ha='center')

plt.title("Effect Size (Cohen's d) for GDL vs Other Models")
plt.xlabel("Comparison")
plt.ylabel("Cohen's d")
plt.xticks(rotation=45, ha='right')


plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# renkleri belirle
colors = []
for val in raw_stats["Cohen's d"]:
    if val > 2:
        colors.append("red")
    elif val > 1:
        colors.append("orange")
    else:
        colors.append("lightgray")

plt.figure(figsize=(12, 5))

# 🔴 1. BARPLOT
ax = sns.barplot(
    data=raw_stats,
    x='Comparison',
    y="Cohen's d",
    palette=colors
)

# 🔴 2. WRITE VALUES
for i, v in enumerate(raw_stats["Cohen's d"]):
    ax.text(i, v + 0.5, f"{v:.2f}", ha='center')

# 🔴 3. EFFECT SIZE LINES (HERE!)
plt.axhline(0.8, linestyle='--', color='gray', label='Medium effect')
plt.axhline(2.0, linestyle='--', color='black', label='Large effect')

plt.legend()

# 🔴 4. GENEL AYARLAR
plt.title("Effect Size (Cohen's d) for GDL vs Other Models")
plt.xlabel("Comparison")
plt.ylabel("Cohen's d")
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# renkleri belirle
colors = []
for model in mean_rank_df['Model']:
    if model == "GDL":
        colors.append("red")              # en iyi model
    elif model in ["GRU", "RF", "SVM"]:
        colors.append("orange")           # strong models
    else:
        colors.append("lightblue")        # others

plt.figure(figsize=(10, 5))
sns.barplot(data=mean_rank_df, x='Model', y='Rank', palette=colors)

plt.title('Mean Model Ranks Based on ROC-AUC Across 5 Folds')
plt.xlabel('Model')
plt.ylabel('Mean Rank (Lower is Better)')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# renkleri belirle
colors = []
for model in mean_rank_df['Model']:
    if model == "GDL":
        colors.append("red")
    elif model in ["GRU", "RF", "SVM"]:
        colors.append("orange")
    else:
        colors.append("lightblue")

plt.figure(figsize=(10, 5))

# 🔴 1. BARPLOT
ax = sns.barplot(data=mean_rank_df, x='Model', y='Rank', palette=colors)

# 🔴 2. WRITE VALUESDIR
for i, v in enumerate(mean_rank_df['Rank']):
    ax.text(i, v + 0.1, f"{v:.2f}", ha='center')

# 🔴 3. GDL'yi BOLD YAP
for tick_label in ax.get_xticklabels():
    if tick_label.get_text() == "GDL":
        tick_label.set_fontweight('bold')

# 🔴 4. GENEL AYARLAR
plt.title('Mean Model Ranks Based on ROC-AUC Across 5 Folds')
plt.xlabel('Model')
plt.ylabel('Mean Rank (Lower is Better)')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# renkleri belirle
colors = []
for model in mean_rank_df['Model']:
    if model == "GDL":
        colors.append("red")
    elif model in ["GRU", "RF", "SVM"]:
        colors.append("orange")
    else:
        colors.append("lightgrey")

plt.figure(figsize=(10, 5))

# 🔴 1. BARPLOT
ax = sns.barplot(data=mean_rank_df, x='Model', y='Rank', palette=colors)

# 🔴 2. WRITE VALUESDIR
for i, v in enumerate(mean_rank_df['Rank']):
    ax.text(i, v + 0.1, f"{v:.2f}", ha='center')

# 🔴 3. GDL'yi BOLD YAP
for tick_label in ax.get_xticklabels():
    if tick_label.get_text() == "GDL":
        tick_label.set_fontweight('bold')

# 🔴 4. GENEL AYARLAR
plt.title('Mean Model Ranks Based on ROC-AUC Across 5 Folds')
plt.xlabel('Model')
plt.ylabel('Mean Rank (Lower is Better)')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 5. CSV outputs

This cell saves CSV files for easy transfer into the manuscript.


In [ ]:
summary_df.to_csv('model_summary_12_models.csv', index=False)
raw_stats.to_csv('gdl_vs_11_models_raw_stats.csv', index=False)
paper_table.to_csv('gdl_vs_11_models_paper_table.csv', index=False)
print('Saved files:')
print('- model_summary_12_models.csv')
print('- gdl_vs_11_models_raw_stats.csv')
print('- gdl_vs_11_models_paper_table.csv')
